In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from statsmodels.nonparametric.smoothers_lowess import lowess

# 读取 charge2 文件夹中的 估计结果.xlsx 文件
file_path = './charge2/估计结果.xlsx'  # 修改文件路径为 charge2 文件夹
df = pd.read_excel(file_path, engine='openpyxl')

# 处理函数：用来处理每一列（内阻，极化电阻，极化电容）
def process_column(column_data, column_name):
    # 检查数据中的 NaN 或无效值并去除
    if np.any(np.isnan(column_data)) or np.any(np.isinf(column_data)):
        print(f"Data in {column_name} contains NaN or Inf, replacing with nearby values")
        for i in range(1, len(column_data) - 1):
            if np.isnan(column_data[i]) or np.isinf(column_data[i]):
                # 替换 NaN 或 Inf 为相邻值的平均值
                column_data[i] = np.mean([column_data[i - 1], column_data[i + 1]])

    # 使用箱型图方法去除异常值
    Q1 = np.percentile(column_data, 25)  # 第一四分位数
    Q3 = np.percentile(column_data, 75)  # 第三四分位数
    IQR = Q3 - Q1  # 四分位距

    # 定义上下边界
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # 将异常值替换为邻近正常值的平均值
    filtered_data = column_data.copy()
    for i in range(1, len(column_data) - 1):
        if column_data[i] < lower_bound or column_data[i] > upper_bound:
            # 替换异常值为附近的两个正常值的平均值
            filtered_data[i] = np.mean([column_data[i - 1], column_data[i + 1]])

    # 应用 RLowess 方法进行平滑
    smoothed = lowess(filtered_data, np.arange(len(filtered_data)), frac=0.6, it=15)  # frac 是平滑参数，it 是迭代次数

    # 提取平滑后的 y 值
    smoothed_data = smoothed[:, 1]

    # 将去噪后的数据保存到新列
    df[column_name + ' Smoothed'] = smoothed_data

    return smoothed_data  # 返回平滑后的数据

# 处理三列数据：内阻，极化电阻，极化电容
columns_to_process = ['内阻 (Ω)', '极化电阻 (Ω)', '极化电容 (F)']
for col in columns_to_process:
    smoothed_data = process_column(df[col].values, col)

# 绘制三个对象的平滑结果
plt.figure(figsize=(15, 12))

# 设置字体，确保显示中文
plt.rcParams['font.sans-serif'] = ['SimHei']  # 黑体字体支持中文
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

for i, col in enumerate(columns_to_process, 1):
    plt.subplot(1, 3, i)
    original_data = df[col].values
    smoothed_data = df[col + ' Smoothed'].values

    # 绘图
    plt.scatter(np.arange(len(original_data)), original_data, label='Original Data', color='gray', alpha=0.3)
    plt.plot(np.arange(len(smoothed_data)), smoothed_data, label='Smoothed Data (Lowess)', color='blue', linewidth=3)

    # 图表标签和标题
    plt.xlabel('Index', fontsize=12)
    plt.ylabel(f'{col} (Ω or F)', fontsize=12)
    plt.title(f'{col} with Lowess Smoothing', fontsize=14)
    plt.legend()

    # 自动调整 Y 轴范围
    plt.autoscale(axis='y', tight=True)

plt.tight_layout()
plt.show()

# 保存结果到 charge2 文件夹中的 Excel 文件
output_path = './charge2/Processed_Results_with_Fitting.xlsx'  # 修改保存路径
df.to_excel(output_path, index=False)

print(f"Results saved to {output_path}")
